In [4]:
# Standard library
import random
import matplotlib.pyplot as plt
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)
from sklearn.model_selection import (
    KFold)

import math

# Custom utilities
from sklearn.preprocessing import StandardScaler

import utils.cross_validation as cval
from utils.model_utils import data_processing_v2, train_test_split_v2

import xgboost as xgb
import shap
from matplotlib.patches import Patch

from math import radians, sin, cos, sqrt, asin


/home/qli/Projects/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions 

In [ ]:

def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two points in km"""
    R = 6371  # Earth's radius in km
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    return R * c

def evaluate_xgboost(model, dtrain, dtest, X_test, y_test, feature_names=None, 
                     biome_name=None, figsize=(12, 14)):
    """
    Comprehensive XGBoost model evaluation with hexbin scatter plot (top) 
    and SHAP feature importance (bottom).
    
    Parameters:
    - model: trained XGBoost model
    - dtrain: training DMatrix (for SHAP explainer)
    - dtest: test DMatrix (for SHAP explainer) 
    - X_test: test features (pandas DataFrame or numpy array)
    - y_test: test targets
    - feature_names: (optional) feature names for SHAP plot
    - biome_name: (optional) name for title
    - show_shap: whether to show SHAP importance
    - figsize: overall figure size (width, height)
    
    Returns:
    - fig: matplotlib figure
    - metrics: dict with MAE, R2, RMSE scores
    """
    
    # Make predictions
    y_pred = model.predict(dtest)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"=== XGBoost Model Performance ===")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    

    fig, ax_scatter = plt.subplots(1, 1, figsize=(figsize[0], figsize[1]//1.5))
    
    # --- HEXBIN SCATTER PLOT (TOP) ---
    hb = ax_scatter.hexbin(y_test, y_pred, gridsize=40, cmap='vanimo', mincnt=1)
    cbar = fig.colorbar(hb, ax=ax_scatter, label='Count')
    cbar.ax.tick_params(labelsize=9)
    
    # Perfect fit line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax_scatter.plot([min_val, max_val], [min_val, max_val], 'darkolivegreen', lw=2, label='Perfect fit')
    
    # Add 10% error bands (optional)
    ax_scatter.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'gray', lw=1, alpha=0.5, linestyle=':')
    ax_scatter.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'gray', lw=1, alpha=0.5, linestyle=':')
    
    ax_scatter.set_xlabel('Actual', fontsize=11)
    ax_scatter.set_ylabel('Predicted', fontsize=11)
    title = f'XGBoost: Predicted vs Actual'
    if biome_name:
        title += f' - {biome_name}'
    ax_scatter.set_title(title, fontsize=13, fontweight='bold')
    
    # Add metrics text box
    metrics_text = f'MAE: {mae:.4f}\nR²: {r2:.4f}\nRMSE: {rmse:.4f}\nn: {len(y_test)}'
    ax_scatter.text(0.05, 0.95, metrics_text, transform=ax_scatter.transAxes,
                    fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax_scatter.legend(loc='lower right', fontsize=9)
    ax_scatter.grid(True, alpha=0.3)
    # Return metrics
    metrics = {
        'MAE': mae,
        'R2': r2,
        'RMSE': rmse,
        'n_samples': len(y_test)
    }
    
    return fig, metrics

def select_one_point_remove_buffer(df, 
                                   buffer_km=50, 
                                   lat_col='lat', lon_col='lon', 
                                   random_seed=None):
    """
    Select exactly ONE point and remove ALL other points within buffer_km.
    
    Parameters:
    - df: DataFrame with coordinates
    - buffer_km: radius in km to remove points (default: 50)
    - lat_col: latitude column name
    - lon_col: longitude column name
    - random_seed: random seed for reproducibility
    - method: 'random' or 'first' or index
    
    Returns:
    - selected_df: DataFrame with 1 selected point
    - remaining_df: DataFrame with points > buffer_km away (kept)
    - removed_df: DataFrame with points <= buffer_km away (removed)
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    

    selected_idx = np.random.choice(df.index, 1, replace=False)[0]
 
    # Get selected point coordinates
    selected_point = df.loc[selected_idx]
    lat_sel = selected_point[lat_col]
    lon_sel = selected_point[lon_col]
    
    # Classify all points
    selected_points = [selected_idx]
    removed_points = []
    remaining_points = []
    
    for idx in df.index:
        if idx == selected_idx:
            continue
        
        # Calculate distance to selected point
        dist = haversine(lat_sel, lon_sel, 
                        df.loc[idx, lat_col], 
                        df.loc[idx, lon_col])
        
        if dist <= buffer_km:
            removed_points.append(idx)  # Too close → remove
        else:
            remaining_points.append(idx)  # Far enough → keep
    
    # Create DataFrames
    selected_df = df.loc[selected_points].copy()
    removed_df = df.loc[removed_points].copy()
    remaining_df = df.loc[remaining_points].copy()
    
    print(f"✓ Removed {len(removed_df)} points within {buffer_km}km")
    
    return selected_df, remaining_df, removed_df



def select_points_stratified(df, n_per_biome=5, biome_col='biome', buffer_km=50,
                            lat_col='lat', lon_col='lon', random_seed=None):
    """
    Select exactly n_per_biome points from each biome.
    
    Parameters:
    - df: DataFrame with coordinates
    - n_per_biome: number of points to select per biome
    - biome_col: column name for biome
    - lat_col: latitude column name
    - lon_col: longitude column name
    - random_seed: random seed for reproducibility
    
    Returns:
    - selected_data: DataFrame with selected points
    - selected_indices: indices of selected points
    - biomes_sampled: dict with biome names and number sampled
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Get biomes present
    biomes = df[biome_col].unique() #This is correct
    selected_indices = []
    biomes_sampled = {}
    
    for biome in biomes:
        # Get indices for this biome
        biome_indices = df[df[biome_col] == biome].index.tolist()
        
        if len(biome_indices) < n_per_biome:
            print(f"Warning: Biome '{biome}' has only {len(biome_indices)} points, "
                  f"sampling all {len(biome_indices)}")
            n_to_sample = len(biome_indices)
        else:
            n_to_sample = n_per_biome
        
        # Randomly select points
        selected = np.random.choice(biome_indices, n_to_sample, replace=False)
        selected_indices.extend(selected)
        biomes_sampled[biome] = len(selected)
    
    # Get selected data
    selected_data = df.loc[selected_indices].copy()
    # Get unique coordinates (if needed)
    selected_coords = selected_data[[lat_col, lon_col]].drop_duplicates().reset_index(drop=True)
    all_coords = df[[lat_col, lon_col]].drop_duplicates().reset_index(drop=True)

    selected_points = []
    removed_points = []
    remaining_points = []

    for idx, row in all_coords.iterrows():
        lat = row[lat_col]
        lon = row[lon_col]
        
        # If this point IS selected, keep it
        if idx in selected_indices:
            selected_points.append(idx)
            continue
        
        # Check distance to ALL selected points
        within_buffer = False
        for _, sel_row in selected_coords.iterrows():
            dist = haversine(lat, lon, sel_row[lat_col], sel_row[lon_col])
            if dist <= buffer_km:
                within_buffer = True
                break  # No need to check more, it's within buffer of at least one
        
        if within_buffer:
            removed_points.append(idx)  # Too close to a selected point → remove
        else:
            remaining_points.append(idx)  # Far enough from all selected points → keep

    # Create DataFrames
    selected_df = df.iloc[selected_points].copy()
    removed_df = df.iloc[removed_points].copy()
    remaining_df = df.iloc[remaining_points].copy()
    
    return selected_df, remaining_df, removed_df

def train_test_split_v3(df, random_key, n_points_to_select=2, buffer_km=100):
    sub_df = df

    # selected, remaining, removed = select_points_stratified(sub_df, 
    #     n_per_biome=n_points_to_select,
    #     buffer_km=buffer_km,
    #     lat_col='lat',
    #     lon_col='lon',
    #     random_seed=random_key
    # )
    selected, remaining, removed = select_one_point_remove_buffer(df, 
                                   buffer_km=50, 
                                   lat_col='lat', lon_col='lon', 
                                   random_seed=random_key)
    

    test = selected.drop(columns=['lat', 'lon', 'biome', "BHAGE" ])
    test_biomes=selected[['biome']]
    train= remaining.drop(columns=['lat', 'lon', 'biome', "BHAGE"])

    y=['transformed npp'] 
    X_train=train.drop(columns=y+['PID'])
    y_train = train['transformed npp'].values

    X_test= test.drop(columns=y+['PID' ])
    y_test = test['transformed npp'].values
    
    scaler= StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    return X_train, y_train, X_test, y_test, test_biomes, scaler

def train_test_split_v2(df, random_key, n_points_to_select=50, buffer_km=100):
    sub_df = df.drop(columns=['biome', 'BHAGE' ])

    selected, remaining, removed = cval.select_points_with_buffer(
        sub_df, 
        n_points=n_points_to_select,
        buffer_km=buffer_km,
        lat_col='lat',
        lon_col='lon',
        random_seed=random_key
    )

    test = selected.drop(columns=['lat', 'lon'])
    train= remaining.drop(columns=['lat', 'lon'])

    y=['transformed npp'] 
    X_train=train.drop(columns=y+['PID'])
    y_train = train['transformed npp'].values

    X_test= test.drop(columns=y+['PID'])
    y_test = test['transformed npp'].values
    
    scaler= StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    return X_train, y_train, X_test, y_test


## Prep

In [7]:
biome_mapping = {
    'Temperate broadleaf forests': 'Temperate broadleaf forests',
    'Temperate conifer forests': 'Temperate conifer forests',
    'Temperate grasslands': 'Temperate grasslands',
    'Xeric shrublands': 'Xeric shrublands',
    'Mediterranean woodlands': 'Mediterranean woodlands',
    'Flooded grasslands': 'Flooded grasslands',
    'Mangroves': 'Mangroves',
    'Tundra': 'Boreal and Tundra forests',
    'Boreal forests or taiga': 'Boreal and Tundra forests',
    'Tropical grasslands': 'Tropical',
    'Tropical coniferous forests': 'Tropical',
    'Tropical moist broadleaf forests': 'Tropical',
    'Tropical dry broadleaf forests': 'Tropical',
    np.nan: np.nan,  # Will be dropped
}


biome_list=["Temperate broadleaf forests", "Temperate conifer forests", "Temperate grasslands",
             "Xeric shrublands", "Boreal and Tundra forests", "Tropical",
               "Mediterranean woodlands"]


In [8]:
fd_df = pd.read_csv('data/final/final_dataset_with_aridity.csv')

fd_df = fd_df[~fd_df["biome"].isin(["Mangroves", "Flooded grasslands"])]

fd_df.dropna(subset=['Raos_Q', 'Functional_Evenness', 'Soil Moisture',
                     'Species Richness', 'Shannon Diversity', "Simpson's Index", 'biome'], inplace=True)
fd_df.drop(columns=['TPA_UNADJ'], inplace=True)


ecoregions=cval.process_ecoregion("data/Ecoregions/Ecoregions2017.shp")

ecoregions=ecoregions[['ECO_NAME', 'geometry']]

#Preprocessing the data for Random forest regression
# fbiome_dfs= data_processing_v2(fd_df, biome_mapping, ecoregions)

fd_df['biome'] = fd_df['biome'].map(biome_mapping) # Only run this once ! 

biome_dfs = {k: v for k, v in fd_df.groupby('biome')}


## Global Model Validation

In [ ]:
random_keys = random.sample(range(1, 10000), 1)  # Generate 10 random keys for reproducibility

params = {
    "objective": "reg:squarederror",
    'early_stopping_rounds': 20,
    "learning_rate": 0.01,
    "max_depth": 6,
    "min_child_weight": 8,
    "gamma": 0.3,
    "lambda": 2.0,      # reg_lambda → lambda
    "alpha": 0.5,       # reg_alpha → alpha
    "tree_method": "hist",
}
n_rounds =300

r2_scores = []
mape_scores = []
rmse_scores = []
mean_y=[]
y_pred_all=[]
y_actual_all=[]
for i in range(len(random_keys)):
    random_key = random_keys[i]
    fX_train, fy_train, fX_test, fy_test, test_biomes, scaler = train_test_split_v3(fd_df, random_key=random_key, n_points_to_select=1, buffer_km=100)

    # # Create regression matrices
    dtrain_reg = xgb.DMatrix(fX_train, fy_train, enable_categorical=True)
    dtest_reg = xgb.DMatrix(fX_test, fy_test, enable_categorical=True)

    # Create regression matrices
    dtrain_reg = xgb.DMatrix(fX_train, fy_train, enable_categorical=True)
    dtest_reg = xgb.DMatrix(fX_test, fy_test, enable_categorical=True)


    evals = [(dtrain_reg, "train"), (dtest_reg, "validation")]

    model = xgb.train(
    params=params,
    dtrain=dtrain_reg,
    num_boost_round=n_rounds,
    evals=evals
    )
    
    # Make predictions
    y_pred = model.predict(dtest_reg)

    y_pred_all.append(y_pred)
    y_actual_all.append(fy_test)
    mean_y.append(np.mean(fy_test))
    r2_scores.append(r2_score(fy_test, y_pred))
    mape_scores.append(mean_absolute_percentage_error(fy_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(fy_test, y_pred)))
    
    print(f"completed iteration {i+1}")



✓ Removed 218 points within 50km
[0]	train-rmse:0.61123	validation-rmse:1.74132
[1]	train-rmse:0.60691	validation-rmse:1.73217
[2]	train-rmse:0.60264	validation-rmse:1.72311
[3]	train-rmse:0.59843	validation-rmse:1.71413
[4]	train-rmse:0.59427	validation-rmse:1.70508
[5]	train-rmse:0.59015	validation-rmse:1.69596
[6]	train-rmse:0.58608	validation-rmse:1.68717
[7]	train-rmse:0.58206	validation-rmse:1.67832
[8]	train-rmse:0.57809	validation-rmse:1.66946
[9]	train-rmse:0.57418	validation-rmse:1.66071
[10]	train-rmse:0.57031	validation-rmse:1.65216
[11]	train-rmse:0.56647	validation-rmse:1.64345
[12]	train-rmse:0.56269	validation-rmse:1.63479
[13]	train-rmse:0.55895	validation-rmse:1.62635
[14]	train-rmse:0.55527	validation-rmse:1.61819
[15]	train-rmse:0.55164	validation-rmse:1.60981
[16]	train-rmse:0.54805	validation-rmse:1.60151
[17]	train-rmse:0.54451	validation-rmse:1.59369
[18]	train-rmse:0.54101	validation-rmse:1.58590
[19]	train-rmse:0.53755	validation-rmse:1.57784
[20]	train-rmse:0

/home/qli/Projects/env1/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [ ]:
df = pd.DataFrame({
    'y_pred_all': y_pred_all,
    'y_actual_all': y_actual_all,
    "mean_y": mean_y,
    "MAPE_scores": mape_scores,
    "RMSE_scores": rmse_scores
})

# If each value is a list like [1], [2], [3]
# Extract the first (and only) element from each list
df['y_actual_all'] = df['y_actual_all'].apply(lambda x: x[0])
df['y_pred_all'] = df['y_pred_all'].apply(lambda x: x[0])

df.to_csv('global_xgboost_results.csv', index=False)

# Clean the data before hexbin
df_clean = df[['y_actual_all', 'y_pred_all']].copy()

# Force numeric conversion
df_clean['y_actual_all'] = pd.to_numeric(df_clean['y_actual_all'], errors='coerce')
df_clean['y_pred_all'] = pd.to_numeric(df_clean['y_pred_all'], errors='coerce')

figsize=(12, 14)

fig, ax_scatter = plt.subplots(1, 1, figsize=(figsize[0], figsize[1]//1.5))
    
hb = ax_scatter.scatter(df_clean['y_actual_all'], 
                        df_clean['y_pred_all'],
                        color='darkolivegreen',
                        alpha=0.6, 
                        s=30)

# Perfect fit line
min_val = min(df['y_actual_all'].min(), df['y_pred_all'].min())
max_val = max(df['y_actual_all'].max(), df['y_pred_all'].max())
ax_scatter.plot([min_val, max_val], [min_val, max_val], 'darkolivegreen', lw=2, label='Perfect fit')

# Add 10% error bands (optional)
ax_scatter.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'gray', lw=1, alpha=0.5, linestyle=':')
ax_scatter.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'gray', lw=1, alpha=0.5, linestyle=':')

ax_scatter.set_xlabel('Actual', fontsize=15)
ax_scatter.set_ylabel('Predicted', fontsize=15)
title = f'XGBoost: Predicted vs Actual'
ax_scatter.set_title(title, fontsize=20, fontweight='bold')

r2 = r2_score(df_clean['y_actual_all'], df_clean['y_pred_all'])
mape = mean_absolute_percentage_error(df_clean['y_actual_all'], df_clean['y_pred_all']) * 100

# --- ADD TEXT BOX AT BOTTOM ---
metrics_text = f'R² = {r2:.4f}\nMAPE = {mape:.2f}%'
ax_scatter.text(0.8, 0.1, metrics_text, 
                transform=ax_scatter.transAxes,
                fontsize=12, 
                verticalalignment='top',
                horizontalalignment='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

plt.tight_layout()
plt.savefig('global_xgboost_predicted_vs_actual.png', dpi=300)


In [ ]:

# randomly select x points per biome and evaluate model performance for each biome

results = {}

fy_series = pd.Series(fy_test.flatten(), index=fX_test.index, name='transformed npp')
combined = fX_test.join(fy_series).join(test_biomes)

for biome, group in combined.groupby('biome'):   # missing 'group'
    X = group.drop(columns=['transformed npp', 'biome'])  # use group not combined
    y = group[['transformed npp']]                                               # use group not combined
    preds   = model.predict(X)

    results[biome]            = preds
    results[f'{biome}_mae']   = mean_absolute_error(y, preds)
    results[f'{biome}_rmse']  = np.sqrt(mean_squared_error(y, preds))
    results[f'{biome}_mape']  = mean_absolute_percentage_error(y, preds)
    results[f'{biome}_r2']    = r2_score(y, preds)

# Summary
print(f"{'Biome':<40} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R2':>8}")
print("-" * 80)
for biome in combined['biome'].unique():
    print(f"{biome:<40} {results[f'{biome}_mae']:>8.3f} {results[f'{biome}_rmse']:>8.3f} {results[f'{biome}_mape']:>8.3f} {results[f'{biome}_r2']:>8.3f}")

## Local model validation

In [64]:
# Define all biome configurations in a single dictionary
XGB_CONFIGS = {
    'Temperate conifer forests': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.025,
            "tree_method": "approx"
        },
        'n_rounds': 240,
        'early_stopping_rounds': 20,
        'description': 'Default configuration'
    },
    
    'Boreal and Tundra forests': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.05,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 100,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset'
    },
    
    'Xeric shrublands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.05,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 100,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset'
    },
    
    'Temperate grasslands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.05,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 100,
        'early_stopping_rounds': 15,
        'description': 'Regularized with more rounds'
    },
    
    'Mediterranean woodlands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.05,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 100,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset (RF better)'
    },
    
    'Temperate broadleaf forests': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.001,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 3.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 200,
        'early_stopping_rounds': 5,
        'description': 'Very small dataset - low LR'
    },
    
    'Tropical': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.005,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 250,
        'early_stopping_rounds': 30,
        'description': 'Large dataset - more rounds'
    }
}

# Function to get config for a biome
def get_xgb_config(biome_name):
    """Get XGBoost configuration for a specific biome"""
    if biome_name in XGB_CONFIGS:
        return XGB_CONFIGS[biome_name]
    else:
        # Default configuration
        return {
            'params': {
                "objective": "reg:squarederror",
                "learning_rate": 0.05,
                "tree_method": "approx"
            },
            'n_rounds': 100,
            'early_stopping_rounds': 10,
            'description': 'Default configuration'
        }


In [ ]:
XGB_CONFIGS = {
    'Temperate conifer forests': {
        'params': { "objective": "reg:squarederror",
            "learning_rate": 0.05,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 200,
        'early_stopping_rounds': 20,
        'description': 'Default configuration'
    },

    'Boreal and Tundra forests': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.02,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 90,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset'
    },

    'Xeric shrublands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.01,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 100,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset'
    },

    'Temperate grasslands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.01,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 250,
        'early_stopping_rounds': 15,
        'description': 'Regularized with more rounds'
    },

    'Mediterranean woodlands': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.005,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 150,
        'early_stopping_rounds': 10,
        'description': 'Regularized for small dataset (RF better)'
    },

    'Temperate broadleaf forests': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.01,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 3.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 380,
        'early_stopping_rounds': 5,
        'description': 'NA'
    },

    'Tropical': {
        'params': {
            "objective": "reg:squarederror",
            "learning_rate": 0.01,
            'max_depth': 6,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_lambda': 2.0,
            'reg_alpha': 0.5,
            "tree_method": "approx"
        },
        'n_rounds': 330,
        'early_stopping_rounds': 30,
        'description': 'Large dataset - more rounds'
    }
}

for biome in biome_list:
    random_keys = random.sample(range(1, 10000), len(biome_dfs[biome]*0.1))  # Generate random keys for reproducibility

    config = get_xgb_config(biome)
    params = config['params']
    n_rounds = config['n_rounds']


    r2_scores = []
    mape_scores = []
    rmse_scores = []
    mean_y=[]
    y_pred_all=[]
    y_actual_all=[]

    for i in range(len(random_keys)):
        random_key = random_keys[i]
        fX_train, fy_train, fX_test, fy_test, test_biomes, scaler = train_test_split_v3(biome_dfs[biome], random_key=random_key, n_points_to_select=1, buffer_km=50)

        # Create training matrices
        dtrain_reg = xgb.DMatrix(fX_train, fy_train, enable_categorical=True)
        dtest_reg = xgb.DMatrix(fX_test, fy_test, enable_categorical=True)

        evals = [(dtrain_reg, "train"), (dtest_reg, "validation")]

        model = xgb.train(
        params=params,
        dtrain=dtrain_reg,
        num_boost_round=n_rounds,
        )

        # Make predictions
        y_pred = model.predict(dtest_reg)

        y_pred_all.append(y_pred)
        y_actual_all.append(fy_test)
        mean_y.append(np.mean(fy_test))
        r2_scores.append(r2_score(fy_test, y_pred))
        mape_scores.append(mean_absolute_percentage_error(fy_test, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(fy_test, y_pred)))
        
        print(f"completed iteration {i+1}")

    df = pd.DataFrame({
        'y_pred_all': y_pred_all,
        'y_actual_all': y_actual_all,
        "mean_y": mean_y,
        "MAPE_scores": mape_scores,
        "RMSE_scores": rmse_scores
    })

    # If each value is a list like [1], [2], [3]
    # Extract the first (and only) element from each list
    df['y_actual_all'] = df['y_actual_all'].apply(lambda x: x[0])
    df['y_pred_all'] = df['y_pred_all'].apply(lambda x: x[0])

    df.to_csv(f'{biome}_xgboost_results.csv', index=False)

    # Clean the data before hexbin
    df_clean = df[['y_actual_all', 'y_pred_all']].copy()

    # Force numeric conversion
    df_clean['y_actual_all'] = pd.to_numeric(df_clean['y_actual_all'], errors='coerce')
    df_clean['y_pred_all'] = pd.to_numeric(df_clean['y_pred_all'], errors='coerce')

    figsize=(12, 14)

    fig, ax_scatter = plt.subplots(1, 1, figsize=(figsize[0], figsize[1]//1.5))
        
    hb = ax_scatter.scatter(df_clean['y_actual_all'], 
                            df_clean['y_pred_all'],
                            color='darkolivegreen',
                            alpha=0.6, 
                            s=30)

    # Perfect fit line
    min_val = min(df['y_actual_all'].min(), df['y_pred_all'].min())
    max_val = max(df['y_actual_all'].max(), df['y_pred_all'].max())
    ax_scatter.plot([min_val, max_val], [min_val, max_val], 'darkolivegreen', lw=2, label='Perfect fit')

    # Add 10% error bands (optional)
    ax_scatter.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'gray', lw=1, alpha=0.5, linestyle=':')
    ax_scatter.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'gray', lw=1, alpha=0.5, linestyle=':')

    ax_scatter.set_xlabel('Actual', fontsize=15)
    ax_scatter.set_ylabel('Predicted', fontsize=15)
    title = f'XGBoost: Predicted vs Actual'
    ax_scatter.set_title(title, fontsize=20, fontweight='bold')

    r2 = r2_score(df_clean['y_actual_all'], df_clean['y_pred_all'])
    mape = mean_absolute_percentage_error(df_clean['y_actual_all'], df_clean['y_pred_all']) * 100

    # --- ADD TEXT BOX AT BOTTOM ---
    metrics_text = f'R² = {r2:.4f}\nMAPE = {mape:.2f}%'
    ax_scatter.text(0.8, 0.1, metrics_text, 
                    transform=ax_scatter.transAxes,
                    fontsize=12, 
                    verticalalignment='top',
                    horizontalalignment='center',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

    plt.tight_layout()
    plt.savefig(f'{biome}_xgboost_predicted_vs_actual.png', dpi=300)

        
            